# K513 · Week 6, Session 2
## Choosing the operating point

On Tuesday you compared three models at one setting. Today you take one model and look at every setting it could have had — and let the campaign's budget decide which one you use.

Still no new algorithm. The same logistic regression and the same tree, one new idea, and four new function calls.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".


One more, specific to today: ask an AI for "the best threshold" and it will pick the one with the highest F1 score. F1 assumes both kinds of mistake cost the same. On this campaign they differ by 49 times, and nothing in the code says so.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one.

---

## 1. Set up

The same models as Tuesday, rebuilt for you so the two sessions sit on one split. Run all three
cells; nothing to fill in.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score,
                             precision_score, roc_curve, auc, roc_auc_score)

LOAN_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/loan.csv"
SEED = 42

In [ ]:
loan_df = pd.read_csv(LOAN_URL)

X = loan_df[['Income', 'CDAccount']]
y = loan_df['PersonalLoan']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['Income']),
    ('bin', 'passthrough', ['CDAccount'])])

baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=SEED))]).fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)

print(f"{len(X_test)} test customers, {y_test.sum()} of them acceptors")

### Two helpers, given complete

You are not expected to be able to write either of these. Read them — especially the last column
of `top_n`, which is the one that matters in section 2.

In [ ]:
def sweep_cuts(probabilities, cuts):
    """One row per cut: how many you mail, what it costs at $10 each,
    how many accept, recall and precision."""
    rows = []
    for c in cuts:
        pred = (probabilities >= c).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
        rows.append({'cut': c, 'mailed': int(pred.sum()),
                     'cost': 10 * int(pred.sum()), 'accepted': int(tp),
                     'recall': recall_score(y_test, pred, zero_division=0),
                     'precision': precision_score(y_test, pred, zero_division=0),
                     'accuracy': accuracy_score(y_test, pred)})
    return pd.DataFrame(rows).round(3)


def top_n(probabilities, n, seed=0):
    """Mail the n highest-scoring customers. Ties at the cut are broken at random,
    which is exactly what a real campaign has to do."""
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(probabilities)), -probabilities))
    chosen = order[:n]
    cut_score = probabilities[order[n - 1]]
    return {'mailed': n,
            'accepted': int(y_test.values[chosen].sum()),
            'precision': y_test.values[chosen].mean().round(3),
            'score at the cut': round(float(cut_score), 4),
            'customers tied there': int((probabilities == cut_score).sum())}

print("helpers ready")

---

## ✏️ Now You Try · 1 — pick a cut and defend it

**About 12 minutes.**

`predict()` is not a separate thing the model does. It is `predict_proba()` with a cut at 0.5
already applied — and 0.5 is a default, not a decision.

```python
y_pred = (probabilities >= 0.5).astype(int)
```

**On Tuesday the default cut mailed 73 customers — \$730 of a \$1,250 budget — and reached 44 of
the 120 acceptors.** It left \$520 of approved money unspent. This block is about going and
getting it.

### (a) The scores behind the predictions

`predict_proba()` returns two columns: the chance of class 0 and the chance of class 1. We want
the second, so `[:, 1]`.

In [ ]:
probs_logreg = logreg.predict_proba(X_test)[:, ____]

print("The first ten customers' estimated chance of accepting:")
print(np.round(probs_logreg[:10], 3))

Check that the default cut really is 0.5 — these two should be identical.

In [ ]:
print("agree on all 1,250 customers:",
      ((probs_logreg >= 0.5).astype(int) == logreg.predict(X_test)).all())

### (b) Sweep the cut

Fill in the blank with the list of cuts to try.

In [ ]:
sweep = sweep_cuts(probs_logreg, ____)
sweep

**Read the `accuracy` column before you read anything else.** It barely moves while
`recall` runs from nearly 1 down to almost nothing.

### (c) Which cut spends the budget?

The budget is \$1,250 and a mailing costs \$10, so it pays for exactly 125 mailings. Read
the `cost` column: which row spends the budget?

Write the cut you chose into the next cell.

In [ ]:
my_cut = ____

pred_at_my_cut = (probs_logreg >= my_cut).astype(int)
print(confusion_matrix(y_test, pred_at_my_cut))

### (d) At your cut, what does the campaign look like?

How much of the budget do you spend, how many acceptors do you reach, and what does the bank
make?

In [ ]:
BUDGET       = 1250   # dollars approved, scaled to our 1,250 test customers
PER_MAILING  = 10
PER_ACCEPTOR = 500

tn, fp, fn, tp = confusion_matrix(y_test, pred_at_my_cut, labels=[0, 1]).ravel()
mailed = int(pred_at_my_cut.sum())
cost   = PER_MAILING * mailed
profit = PER_ACCEPTOR * tp - cost

print(f"cut               {my_cut}")
print(f"mailed            {mailed}")
print(f"spent             ${cost:,} of ${BUDGET:,}   ({cost / BUDGET:.0%} of the budget)")
print(f"acceptors reached {tp} of 120")
print(f"wasted mailings   {fp}")
print(f"profit            ${profit:,}")

print(f"\nAt the default cut of 0.5 the same model spent $730 and made $21,270.")

### (e) One sentence

You have just moved a number that scikit-learn chose for you, and the campaign's profit moved with
it. What would have to change about the bank's costs for you to move the cut **up** instead?

> ✏️ **Not collected.** Be ready to say it out loud.

---

## 2. Which 125?

The \$1,250 buys 125 mailings whatever you do. The only question is who is on the list.

In [ ]:
probs_tree = tree.predict_proba(X_test)[:, 1]
K = 125   # one in ten of the 1,250 test customers

print("Rank by the logistic model :", top_n(probs_logreg, K))
print("Rank by the tree           :", top_n(probs_tree, K))
print(f"Mail {K} at random         : about {int(K * y_test.mean())} accept")

| Same \$1,250, same 125 mailings | Acceptors reached | Profit |
|---|---|---|
| Mail 125 at random — no model | 12 | \$4,750 |
| Mail the 125 the model picks | **68** | **\$32,750** |

**Nearly seven times the profit for the same money.** None of it came from the accuracy score,
which never moved. All of it came from the ranking, which was sitting inside `predict_proba` the
whole time.

Profit is \$500 for every acceptor reached minus \$10 for everyone mailed. Both rows mail 125
people and spend the same \$1,250 — the whole difference is who is on the list.

**And the tree?** It cannot reliably pick 125. Look at `customers tied there` in the output above:
151 customers share the score that decides its 125th place, so it has to take 93 of them at
random. Change the `seed` and it reaches anywhere from 54 to 73 acceptors — \$25,750 to \$35,250
of profit, from nothing but which tied customer got picked. The logistic model returns 68 every
time. That is what section (b) below is about.

---

## ✏️ Now You Try · 2 — read the curves

**About 8 minutes.**

An ROC curve draws one point for every cut you could have chosen. Up the side is the share of real
acceptors you reach — recall. Along the bottom is the share of the customers who would have said no
that you mailed anyway — the false alarm rate.

**Every row of the sweep you built in section 1 is a dot on that line.** The code below marks four
of them, so you can see where the default cut put you and where the budget put you.

**AUC** is the area underneath. Take one acceptor and one non-acceptor at random: AUC is the chance
the model gave the acceptor the higher score. It scores the *ranking* — which is what lets you
compare two models before anybody has chosen a cut. It tells you nothing about the cut you will
actually use.

### (a) Draw both curves and print both AUCs

Two blanks. `roc_curve` takes what really happened first and the **probabilities** second — not
the predictions.

In [ ]:
plt.figure(figsize=(7, 5))

for probs, name in [(probs_logreg, 'Logistic regression'),
                    (probs_tree, 'Decision tree')]:
    fpr, tpr, cuts = roc_curve(____, ____)
    plt.plot(fpr, tpr, marker='o', ms=3, label=f'{name}: AUC {auc(fpr, tpr):.3f}')

# Mark where a few of the cuts from your sweep sit on the logistic curve
fpr, tpr, cuts = roc_curve(y_test, probs_logreg)
for c in [0.05, 0.15, 0.28, 0.50]:
    i = int(np.argmin(np.abs(cuts - c)))
    plt.plot(fpr[i], tpr[i], 'o', ms=10, color='#B26B00', zorder=5)
    plt.annotate(f'cut {c:.2f}', (fpr[i], tpr[i]), xytext=(fpr[i] + 0.04, tpr[i] - 0.05),
                 color='#B26B00', fontweight='bold')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='A model that knows nothing: 0.500')
plt.xlabel('false positive rate (FPR)')
plt.ylabel('recall (TPR)')
plt.legend(loc='lower right')
plt.show()

`roc_auc_score()` is the short form — it takes the probabilities directly and gives the
same number.

In [ ]:
print(f"Logistic regression {roc_auc_score(y_test, probs_logreg):.4f}")
print(f"Decision tree       {roc_auc_score(y_test, probs_tree):.4f}")

### (b) Why does the tree's curve have corners?

Count how many different scores each model gives the 1,250 test customers.

In [ ]:
print("Logistic regression:", len(np.unique(probs_logreg)), "different scores")
print("Decision tree      :", len(np.unique(probs_tree)), "different scores")
print()
print("The tree's whole vocabulary:", np.round(np.unique(probs_tree), 3))

> ✏️ **Not collected.** In one sentence: what do the corners have to do with that count?

### (c) One sentence

The AUCs are about 0.94 and 0.93 — nearly the same. At the budget the logistic model reached
**68** of the 120 acceptors; the tree reached **59** on the run above, and anywhere from 54 to 73
depending on how its ties fall.

Why is the gap in business wider than the gap in AUC?

> ✏️ **Not collected.** This is the question worth arguing about.

---

## Choosing a metric

| What the decision needs | Report |
|---|---|
| A yes or no, and both mistakes cost the same | Accuracy |
| Do not miss the ones that matter | Recall |
| Do not act on the wrong ones | Precision |
| A shortlist of fixed length | Precision at that cut |
| No cut chosen yet — are we still comparing models? | AUC |

Question first, cost second, metric third. The metric is a business decision; if nobody makes it
deliberately it gets made by whoever wrote the code, usually by accident, usually as accuracy.

**The weekly homework covers both sessions** and is on Canvas as
*Week 6 Homework - Model Evaluation*.